# 02 - Preprocesamiento de Texto

Este notebook implementa el pipeline de preprocesamiento para los datasets de phishing y smishing.

## Contenido
1. Carga de datos
2. Limpieza de texto
3. Normalización
4. Tokenización
5. Comparación antes/después
6. Guardado de datos preprocesados

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from tqdm import tqdm

from src.data import SMSLoader, EmailLoader, TextPreprocessor
from src.data.preprocessor import get_preprocessor, EmailPreprocessor, SMSPreprocessor

import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos

In [ ]:
# Cargar dataset SMS
sms_loader = SMSLoader(data_dir='../data')

try:
    sms_df = sms_loader.load_uci_sms_spam()
except FileNotFoundError:
    from src.data.sms_loader import create_sample_dataset
    sms_df = create_sample_dataset()

print(f"Dataset SMS: {len(sms_df)} mensajes")
sms_df.head()

## 2. Configuración del Preprocesador

In [ ]:
# Crear preprocesador para modelos clásicos de ML
preprocessor_ml = get_preprocessor(
    data_type='sms',
    for_classical_ml=True
)

# Crear preprocesador para transformers (mínimo procesamiento)
preprocessor_transformers = get_preprocessor(
    data_type='sms',
    for_transformers=True
)

print("Configuración preprocesador ML:")
print(f"  - lowercase: {preprocessor_ml.lowercase}")
print(f"  - remove_stopwords: {preprocessor_ml.remove_stopwords}")
print(f"  - lemmatize: {preprocessor_ml.lemmatize}")

## 3. Ejemplos de Preprocesamiento

In [ ]:
# Ejemplo de preprocesamiento
sample_texts = [
    "CONGRATULATIONS! You've won £1000! Claim NOW: http://bit.ly/claim",
    "Hi! Meeting at 3pm tomorrow. Let me know if that works.",
    "URGENT: Your bank account has been LOCKED! Call 1-800-123-4567",
]

print("Comparación de preprocesamiento:\n")
for text in sample_texts:
    print(f"Original:    {text}")
    print(f"ML Classic:  {preprocessor_ml.preprocess(text)}")
    print(f"Transformer: {preprocessor_transformers.preprocess(text)}")
    print()

## 4. Extracción de Características Adicionales

In [ ]:
# Extraer características antes del preprocesamiento
sample_text = "URGENT! Click http://suspicious-link.com to verify your $500 prize! Call 1-800-SCAM"

features = preprocessor_ml.extract_features_count(sample_text)
print("Características extraídas del texto original:")
for name, value in features.items():
    print(f"  {name}: {value}")

## 5. Preprocesamiento del Dataset Completo

In [ ]:
# Preprocesar todo el dataset
print("Preprocesando dataset SMS...")

# Para ML clásico
sms_df['text_clean_ml'] = preprocessor_ml.preprocess_batch(
    sms_df['body'], 
    show_progress=True
)

# Para transformers
sms_df['text_clean_transformer'] = preprocessor_transformers.preprocess_batch(
    sms_df['body'],
    show_progress=True
)

print("\nPreprocesamiento completado!")

In [ ]:
# Ver resultados
print("Ejemplos de preprocesamiento:")
for idx in range(min(5, len(sms_df))):
    print(f"\n[{idx}] Original: {sms_df.iloc[idx]['body'][:80]}...")
    print(f"    Limpio ML: {sms_df.iloc[idx]['text_clean_ml'][:80]}...")

## 6. Estadísticas Post-Preprocesamiento

In [ ]:
# Comparar longitudes antes y después
sms_df['original_length'] = sms_df['body'].str.len()
sms_df['clean_length'] = sms_df['text_clean_ml'].str.len()

print("Comparación de longitudes:")
print(f"\nLongitud original (promedio): {sms_df['original_length'].mean():.1f}")
print(f"Longitud limpia (promedio): {sms_df['clean_length'].mean():.1f}")
print(f"Reducción promedio: {(1 - sms_df['clean_length'].mean()/sms_df['original_length'].mean())*100:.1f}%")

## 7. Guardado de Datos Preprocesados

In [ ]:
# Guardar dataset preprocesado
output_cols = ['body', 'label', 'source', 'text_clean_ml', 'text_clean_transformer']
output_df = sms_df[output_cols]

sms_loader.save_processed(output_df, filename='sms_preprocessed.csv')
print("Dataset preprocesado guardado exitosamente!")

## 8. Resumen

En este notebook hemos:

1. ✅ Cargado los datasets de SMS
2. ✅ Configurado preprocesadores para diferentes tipos de modelos
3. ✅ Aplicado limpieza y normalización de texto
4. ✅ Extraído características adicionales
5. ✅ Guardado los datos preprocesados

### Próximos pasos:
- Feature Engineering (notebook 03)
- Entrenamiento de modelos (notebook 04)